# 배우별 500만 돌파 확률 분석

로지스틱 회귀를 사용해 배우 출연 여부와 500만 관객 돌파 여부의 연관 신호를 분석합니다. 정확도만 보지 않고 정밀도, 재현율, F1, ROC-AUC를 함께 확인합니다.

In [ ]:
from pathlib import Path
import re
import warnings

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_STATE = 42


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "data").exists() and (path / "notebooks").exists():
            return path
    return current.parent if current.name == "notebooks" else current


ROOT = find_project_root()


def configure_korean_font() -> None:
    preferred_fonts = ["AppleGothic", "NanumGothic", "NanumBarunGothic", "Malgun Gothic"]
    available_fonts = {font.name for font in fm.fontManager.ttflist}
    for family in preferred_fonts:
        if family in available_fonts:
            plt.rcParams["font.family"] = family
            break
    plt.rcParams["axes.unicode_minus"] = False


configure_korean_font()
ROOT

In [ ]:
DATA_CANDIDATES = [
    ROOT / "data/kobis_tmdb_title_matches.csv",
    ROOT / "data/tmdb_korean_movies_kobis.csv",
    Path("/content/drive/MyDrive/Colab Notebooks/top50_actors_boxoffice.csv"),
]
TITLE_COLUMNS = ["match_title", "title", "movie_name", "Movie Name"]
AUDIENCE_COLUMNS = ["audience_count", "kobis_audi_acc", "Audience", "Admissions"]
CAST_COLUMNS = ["tmdb_cast", "cast", "Actor"]


def read_first_available_csv(candidates):
    for path in candidates:
        if path.exists():
            return pd.read_csv(path, encoding="utf-8-sig"), path
    searched = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(f"분석용 CSV를 찾지 못했습니다. 확인 경로:\n{searched}")


def first_existing_column(frame: pd.DataFrame, candidates, required: bool = True):
    for column in candidates:
        if column in frame.columns:
            return column
    if required:
        raise ValueError(f"필수 컬럼이 없습니다. 후보: {candidates}")
    return None


def clean_actor_name(value: object) -> str:
    text = str(value).strip()
    text = re.sub(r"\([^)]*\)$", "", text).strip()
    return text


def split_cast(value: object) -> list[str]:
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "null"}:
        return []
    parts = text.split("|") if "|" in text else [text]
    names = [clean_actor_name(part) for part in parts]
    return [name for name in names if name]


def prepare_actor_movie_frame(frame: pd.DataFrame) -> pd.DataFrame:
    title_col = first_existing_column(frame, TITLE_COLUMNS, required=False)
    audience_col = first_existing_column(frame, AUDIENCE_COLUMNS)
    cast_col = first_existing_column(frame, CAST_COLUMNS)

    result = pd.DataFrame({
        "movie_title": frame[title_col] if title_col else [f"row_{idx}" for idx in range(len(frame))],
        "audience_count": pd.to_numeric(frame[audience_col], errors="coerce"),
        "actors": frame[cast_col].map(split_cast),
    })
    result = result.dropna(subset=["audience_count"])
    result = result[(result["audience_count"] > 0) & result["actors"].map(bool)].copy()
    result["audience_million"] = result["audience_count"] / 1_000_000
    return result.reset_index(drop=True)


def build_actor_matrix(movies: pd.DataFrame, min_movie_count: int = 3):
    actor_counts = pd.Series(
        [actor for actors in movies["actors"] for actor in actors],
        dtype="object",
    ).value_counts()
    selected_actors = actor_counts[actor_counts >= min_movie_count].index.tolist()
    if not selected_actors:
        raise ValueError("min_movie_count 기준을 만족하는 배우가 없습니다. 기준을 낮춰주세요.")

    matrix = pd.DataFrame(0, index=movies.index, columns=selected_actors, dtype=int)
    actor_set = set(selected_actors)
    for idx, actors in movies["actors"].items():
        for actor in actors:
            if actor in actor_set:
                matrix.at[idx, actor] = 1
    matrix = matrix.loc[:, matrix.sum().sort_values(ascending=False).index]
    return matrix, actor_counts


def actor_summary_table(movies: pd.DataFrame) -> pd.DataFrame:
    exploded = movies.explode("actors")
    return exploded.groupby("actors").agg(
        n_movies=("movie_title", "nunique"),
        mean_audience=("audience_count", "mean"),
        median_audience=("audience_count", "median"),
        hit_rate_3m=("audience_count", lambda values: (values >= 3_000_000).mean()),
        hit_rate_5m=("audience_count", lambda values: (values >= 5_000_000).mean()),
    )


raw_data, data_path = read_first_available_csv(DATA_CANDIDATES)
movies = prepare_actor_movie_frame(raw_data)
X, actor_counts = build_actor_matrix(movies, min_movie_count=3)
actor_stats = actor_summary_table(movies)

print(f"데이터: {data_path}")
print(f"영화 수: {len(movies):,}건")
print(f"분석 대상 배우 수: {X.shape[1]:,}명 (3편 이상 출연 기준)")
movies[["movie_title", "audience_count", "actors"]].head()

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

HIT_THRESHOLD = 5_000_000
movies["is_hit"] = (movies["audience_count"] >= HIT_THRESHOLD).astype(int)
y = movies["is_hit"]

class_counts = y.value_counts().sort_index()
print(f"흥행 기준: {HIT_THRESHOLD:,}명")
print(class_counts.rename(index={0: "미달", 1: "돌파"}))

x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)


def classification_metrics(name, estimator, x_eval, y_eval) -> dict[str, float | str]:
    pred = estimator.predict(x_eval)
    if hasattr(estimator, "predict_proba"):
        prob = estimator.predict_proba(x_eval)[:, 1]
    else:
        prob = None
    return {
        "model": name,
        "accuracy": accuracy_score(y_eval, pred),
        "precision": precision_score(y_eval, pred, zero_division=0),
        "recall": recall_score(y_eval, pred, zero_division=0),
        "f1": f1_score(y_eval, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_eval, prob) if prob is not None and y_eval.nunique() == 2 else np.nan,
    }

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(x_train, y_train)

min_class_train = y_train.value_counts().min()
cv = min(5, int(min_class_train))
logit = LogisticRegressionCV(
    Cs=np.logspace(-2, 2, 12),
    cv=cv,
    scoring="roc_auc" if cv >= 2 else "accuracy",
    class_weight="balanced",
    max_iter=5_000,
    random_state=RANDOM_STATE,
)
logit.fit(x_train, y_train)

metrics = pd.DataFrame([
    classification_metrics("최빈값 기준선", baseline, x_test, y_test),
    classification_metrics("LogisticCV 배우 원-핫", logit, x_test, y_test),
])
metrics

In [ ]:
single_actor_matrix = pd.DataFrame(np.eye(len(X.columns)), columns=X.columns)
single_actor_prob = logit.predict_proba(single_actor_matrix)[:, 1]

actor_signal = pd.DataFrame({
    "actor": X.columns,
    "logit_coef": logit.coef_[0],
    "single_actor_hit_prob": single_actor_prob,
}).join(actor_stats, on="actor")
actor_signal["single_actor_hit_prob_pct"] = actor_signal["single_actor_hit_prob"] * 100
actor_signal["mean_audience_million"] = actor_signal["mean_audience"] / 1_000_000
actor_signal["median_audience_million"] = actor_signal["median_audience"] / 1_000_000

columns = [
    "actor", "n_movies", "logit_coef", "single_actor_hit_prob_pct",
    "hit_rate_5m", "mean_audience_million", "median_audience_million",
]
actor_signal.sort_values("logit_coef", ascending=False)[columns].head(20)

In [ ]:
TEST_ACTORS = ["최민식", "이병헌", "강하늘", "공유", "이정재", "정우성", "송강호", "김윤석"]

simulation = actor_signal[actor_signal["actor"].isin(TEST_ACTORS)].copy()
missing = sorted(set(TEST_ACTORS) - set(simulation["actor"]))
if missing:
    print("데이터에 없거나 3편 이상 출연 기준을 통과하지 못한 배우:", ", ".join(missing))

simulation = simulation.sort_values("single_actor_hit_prob_pct", ascending=False)
simulation[columns]

In [ ]:
plot_data = actor_signal.sort_values("single_actor_hit_prob_pct", ascending=False).head(15)
plot_data = plot_data.sort_values("single_actor_hit_prob_pct")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(plot_data["actor"], plot_data["single_actor_hit_prob_pct"], color="#457b9d")
ax.set_xlim(0, 100)
ax.set_title("배우 신호만 넣었을 때 500만 돌파 확률 상위 배우")
ax.set_xlabel("예측 확률 (%)")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()

## 해석 메모

- `single_actor_hit_prob_pct`는 해당 배우 신호만 켠 모델 입력의 확률입니다. 실제 영화의 장르, 제작비, 감독, 개봉 시기까지 반영한 확률은 아닙니다.
- 클래스 불균형 때문에 정확도보다 F1과 ROC-AUC를 우선 확인합니다.
- 300만/500만 기준을 바꾸면 배우 순위가 달라질 수 있으므로 기준값은 분석 목적에 맞게 고정해야 합니다.